# Brussels Post-Processing Pipeline

Applies post-detection filters to M-DRAC conflicts:
1. Teleportation filter - removes conflicts during tracking glitches
2. Pair deduplication - keeps worst-case detection per pair

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '/home/ubuntu/prem')

# Modular imports
from ssm.utils import load_config
from filters.postprocessing.teleportation_filter import filter_conflicts_by_teleportation
from utils.io_helpers import save_detection_results

print("✓ Imports complete")

✓ Imports complete


## Configuration

In [12]:
# Load configuration
config = load_config("/home/ubuntu/prem/config.yaml")

# Paths
REGION = "brussels"
START_DATE = "2025-06-04"
DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
RESULTS_DIR = "/home/ubuntu/prem/regions/brussels/results"

# Input files (from main pipeline)
MDRAC_INPUT = f"{RESULTS_DIR}/{REGION}/mdrac/04/mdrac_04.csv"

# Output
OUTPUT_DIR = f"{RESULTS_DIR}/{REGION}"

print(f"Configuration:")
print(f"  Region: {REGION}")
print(f"  Date: {START_DATE}")
print(f"  M-DRAC input: {MDRAC_INPUT}")
print(f"  Output: {OUTPUT_DIR}")

Configuration:
  Region: brussels
  Date: 2025-06-04
  M-DRAC input: /home/ubuntu/prem/regions/brussels/results/brussels/mdrac/04/mdrac_04.csv
  Output: /home/ubuntu/prem/regions/brussels/results/brussels


## Utilities

In [13]:
def create_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create normalized pair_id (always smaller ID first).
    Ensures (id1=100, id2=200) and (id1=200, id2=100) are treated as same pair.
    """
    df = df.copy()
    df['pair_id'] = df.apply(
        lambda r: f"{min(r['id1'], r['id2'])}_{max(r['id1'], r['id2'])}",
        axis=1
    )
    return df


def deduplicate_pairs(df: pd.DataFrame, metric_col: str = 'MDRAC') -> pd.DataFrame:
    """
    Group by pair_id and select worst case.
    
    Args:
        df: DataFrame with pair_id column
        metric_col: Column to maximize ('MDRAC' or 'composite_risk')
    
    Returns:
        Deduplicated DataFrame (one row per pair)
    """
    initial_count = len(df)
    unique_pairs = df['pair_id'].nunique()
    
    # Keep worst-case detection per pair
    deduplicated = df.loc[df.groupby('pair_id')[metric_col].idxmax()].copy()
    
    reduction = 100 * (initial_count - len(deduplicated)) / initial_count
    print(f"  Deduplication: {initial_count:,} → {len(deduplicated):,} ({reduction:.1f}% reduction)")
    print(f"  Avg detections per pair: {initial_count/unique_pairs:.1f}")
    
    return deduplicated

---
# M-DRAC Post-Processing

## Load M-DRAC Detections

In [14]:
print("="*70)
print("M-DRAC POST-PROCESSING")
print("="*70)

mdrac_raw = pd.read_csv(MDRAC_INPUT)
print(f"\nLoaded {len(mdrac_raw)} raw M-DRAC detections")
print(f"Columns: {list(mdrac_raw.columns)}")
print(f"\nSample:")
mdrac_raw.head(3)

M-DRAC POST-PROCESSING

Loaded 5 raw M-DRAC detections
Columns: ['timestamp', 'id1', 'id2', 'zone', 'interaction', 'leader', 'dist', 'TTC', 'MDRAC', 'closing_speed', 'speed_diff', 'yaw_diff', 'link']

Sample:


,timestamp,id1,id2,zone,interaction,leader,dist,TTC,MDRAC,closing_speed,speed_diff,yaw_diff,link
0,2025-06-04 04:15:37.974562883,13130564,13130814,C-L1,van_v_car,13130564,6.403067,0.911280,4.118060,7.026452,0.994987,3.580986,https://di-india-collab-2.flow-analytics.io/to...
1,2025-06-04 06:25:52.159018040,13244529,13246303,D-L1,bus_v_car,13246303,6.464994,0.582026,16.149534,11.107735,3.807578,156.248490,https://di-india-collab-2.flow-analytics.io/to...
2,2025-06-04 09:13:50.980880022,13383416,13384465,D-L1,bus_v_car,13384465,5.263956,1.176440,11.164707,4.474478,2.860390,89.860370,https://di-india-collab-2.flow-analytics.io/to...


## 1. Teleportation Filter

In [15]:
print("\n" + "="*70)
print("TELEPORTATION FILTER")
print("="*70)

# Load vehicle data for the same date
vehicle_data_path = f"{DATA_DIR}/{START_DATE}/objects.parquet"
print(f"Loading vehicle data: {vehicle_data_path}")

try:
    df = pd.read_parquet(vehicle_data_path)
    print(f"✓ Loaded {len(df):,} vehicle records")
    
    # Apply teleportation filter
    mdrac_clean = filter_conflicts_by_teleportation(
        conflicts=mdrac_raw,
        vehicle_data=df,
        max_jump=config['postprocessing']['teleportation_filter'].get('max_speed_threshold', 50.0),
        verbose=True
    )
    
    removed = len(mdrac_raw) - len(mdrac_clean)
    print(f"\n✓ Removed {removed} conflicts with teleportation glitches")
    
except FileNotFoundError:
    print(f"⚠️  Vehicle data not found: {vehicle_data_path}")
    print("  Skipping teleportation filter")
    mdrac_clean = mdrac_raw.copy()


TELEPORTATION FILTER
Loading vehicle data: /home/ubuntu/data/uploads/objects/clean/2025-06-04/objects.parquet
⚠️  Vehicle data not found: /home/ubuntu/data/uploads/objects/clean/2025-06-04/objects.parquet
  Skipping teleportation filter


## 2. Pair Deduplication

In [16]:
print("\n" + "="*70)
print("PAIR DEDUPLICATION")
print("="*70)

# Create pair IDs
mdrac_clean = create_pair_id(mdrac_clean)

# Deduplicate (keep worst case per pair)
mdrac_final = deduplicate_pairs(mdrac_clean, metric_col='MDRAC')

print(f"\n✓ Final M-DRAC conflicts: {len(mdrac_final):,}")


PAIR DEDUPLICATION
  Deduplication: 5 → 5 (0.0% reduction)
  Avg detections per pair: 1.0

✓ Final M-DRAC conflicts: 5


## 3. Save Results

In [17]:
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

# Save post-processed results
output_path = save_detection_results(
    mdrac_final,
    OUTPUT_DIR,
    'mdrac_postprocessed',
    REGION,
    START_DATE
)

print(f"\n✓ Saved {len(mdrac_final)} post-processed conflicts")
print(f"  Output: {output_path}")

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"  Raw detections: {len(mdrac_raw):,}")
print(f"  After teleportation filter: {len(mdrac_clean):,}")
print(f"  After deduplication: {len(mdrac_final):,}")
print(f"  Total removed: {len(mdrac_raw) - len(mdrac_final):,} ({100*(len(mdrac_raw)-len(mdrac_final))/len(mdrac_raw):.1f}%)")
print("="*70)


SAVING RESULTS
✓ Saved 5 conflicts to /home/ubuntu/prem/regions/brussels/results/brussels/brussels/mdrac_postprocessed/04/mdrac_postprocessed_04.csv

✓ Saved 5 post-processed conflicts
  Output: /home/ubuntu/prem/regions/brussels/results/brussels/brussels/mdrac_postprocessed/04/mdrac_postprocessed_04.csv

SUMMARY
  Raw detections: 5
  After teleportation filter: 5
  After deduplication: 5
  Total removed: 0 (0.0%)


## Analysis

In [18]:
# Zone distribution
print("\nZone Distribution:")
print(mdrac_final['zone'].value_counts())

# Severity distribution  
print("\nSeverity Distribution:")
severity_bins = [0, 5.0, 7.0, float('inf')]
severity_labels = ['moderate', 'severe', 'critical']
mdrac_final['severity_bin'] = pd.cut(mdrac_final['MDRAC'], bins=severity_bins, labels=severity_labels)
print(mdrac_final['severity_bin'].value_counts())

# Top 10 conflicts
print("\nTop 10 Critical Conflicts:")
mdrac_final.nlargest(10, 'MDRAC')[['timestamp', 'zone', 'interaction', 'MDRAC', 'TTC', 'dist']]


Zone Distribution:
zone
D-L1    2
B-L2    2
C-L1    1
Name: count, dtype: int64

Severity Distribution:
severity_bin
critical    3
moderate    2
severe      0
Name: count, dtype: int64

Top 10 Critical Conflicts:


,timestamp,zone,interaction,MDRAC,TTC,dist
4,2025-06-04 22:46:05.458056927,B-L2,car_v_car,16.474699,0.692245,7.923012
1,2025-06-04 06:25:52.159018040,D-L1,bus_v_car,16.149534,0.582026,6.464994
2,2025-06-04 09:13:50.980880022,D-L1,bus_v_car,11.164707,1.176440,5.263956
3,2025-06-04 14:38:05.939728022,B-L2,van_v_car,4.985370,0.780899,6.231300
0,2025-06-04 04:15:37.974562883,C-L1,van_v_car,4.118060,0.911280,6.403067
